# 🍎 My School Menus - Calendar Generator

This notebook helps you generate iCalendar (.ics) files from your school's lunch and breakfast menus on [My School Menus](https://myschoolmenus.com).

**What this does:**
- Fetches your school's menu data from myschoolmenus.com
- Generates calendar files (.ics) that you can import into Google Calendar, Apple Calendar, Outlook, etc.
- Supports both breakfast and lunch menus
- Lets you customize meal times and output options

**How to use:**
1. Run each cell in order (click the ▶️ play button or press Shift+Enter)
2. Fill in the configuration form with your school's information
3. Run the generation cell to create your calendar files
4. Download the generated .ics files

---
*Based on [my-school-menus](https://github.com/andrewdefilippis/my_school_menus) by Andrew R. DeFilippis*

## 📋 Step 1: Install Dependencies

Run this cell to install the required Python packages.

In [ ]:
print("📦 Installing required packages...")
!pip install -q requests icalendar ipywidgets
print("✅ Installation complete!")

## 📚 Step 2: Import Libraries

Import all necessary libraries and modules.

In [ ]:
import requests
import json
import uuid
from datetime import datetime, time, timedelta, date
from dataclasses import dataclass
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from google.colab import files
import io
import zipfile

print("✅ Libraries imported successfully!")

## 🔧 Step 3: API Classes

These classes handle communication with the My School Menus API.

In [ ]:
DOMAIN = 'myschoolmenus.com'

@dataclass
class RequestParams:
    path: str = None
    headers: dict = None
    exception_message: str = None

class Request:
    def __init__(self):
        pass

    @staticmethod
    def url(path) -> str:
        """Get the URL for the API."""
        return f"https://{DOMAIN}{path}"

    @staticmethod
    def get(params: RequestParams) -> dict:
        """Get a response from the API."""
        url = f"{Request.url(params.path)}"
        response = requests.get(url=url, headers=params.headers)
        if response.status_code != 200:
            raise ValueError(
                f"Endpoint {url} returned status code {response.status_code}: {response.reason}"
            )
        try:
            json = response.json()
            if not json['data']:
                raise ValueError(params.exception_message)
        except requests.exceptions.JSONDecodeError:
            raise ValueError(f"Unable to decode JSON response")
        return response.json()

class Menus:
    def __init__(self):
        self.path = '/api/organizations'

    def get(self, district_id: int, site_id: int = None, menu_id: int = None, date: datetime.date = None) -> dict:
        """Get a menu by district ID, menu ID, and optionally a date."""
        exception_message = f"No menu found for district {district_id}"
        if site_id:
            path = self.path + f"/{district_id}/sites/{site_id}"
        elif menu_id:
            path = self.path + f"/{district_id}/menus/{menu_id}"
            if date:
                path = path + f"/year/{date.strftime('%Y')}/month/{date.strftime('%m')}/date_overwrites"
        else:
            path = self.path
        return Request.get(RequestParams(
            path=path,
            exception_message=exception_message + f", menu {menu_id}{f', and date {date}' if date else ''}" if menu_id else exception_message
        ))

    @staticmethod
    def menu_months(menu: dict) -> list:
        """Get a list of months with available menus."""
        return [datetime.fromisoformat(date) for date in menu['data']['published_months']]

class Organizations:
    def __init__(self):
        self.path = '/api/organizations'

    def get(self, organization_id: int = None) -> dict:
        """Get an organization by organization ID, or return all organizations."""
        return Request.get(RequestParams(
            path=self.path + f"/{organization_id}" if organization_id else self.path,
            exception_message=f"No organization found for organization {organization_id}" if organization_id else ""
        ))

class Sites:
    def __init__(self):
        self.path = 'api/organizations'

    def get(self, district_id: int, site_id: int) -> dict:
        """Get a site for a given district."""
        return Request.get(RequestParams(
            path=self.path + f"/{district_id}/sites/{site_id}",
            exception_message=f"No site found for district {district_id} and site {site_id}"
        ))

print("✅ API classes loaded successfully!")

## 📅 Step 4: Calendar Classes

These classes handle the conversion of menu data to iCalendar format.

In [ ]:
class Calendar:
    def __init__(self, default_breakfast_time=time(8, 0), default_lunch_time=time(12, 0)):
        """Initialize the Calendar with default times for meals."""
        self.default_breakfast_time = default_breakfast_time
        self.default_lunch_time = default_lunch_time

    def events(self, menu: json, menu_type: str = "lunch", include_time: bool = False) -> list:
        """Generate a list of events from a menu."""
        event_list = []
        menu_data = menu['data']
        if not menu_data:
            raise ValueError(f"Missing menu data.")

        event_time = None
        if include_time:
            if menu_type.lower() == "breakfast":
                event_time = self.default_breakfast_time
            else:
                event_time = self.default_lunch_time

        for entry in menu_data:
            if entry is None:
                continue
            
            try:
                prefix = "L: " if menu_type.lower() == "lunch" else "B: "
                
                recipe_count = 0
                category_count = 0
                summary = ""
                description_parts = []
                
                for item in json.loads(entry['setting'])['current_display']:
                    if item['type'] == 'recipe' and recipe_count == 0:
                        recipe_count += 1
                        summary = f"{prefix}{item['name']}"
                    
                    if item['type'] == 'category' and category_count == 0:
                        category_count += 1
                        description_parts.append(f"{item['name']}:")
                    elif item['type'] == 'category':
                        description_parts.append("")
                        description_parts.append(f"{item['name']}:")
                    else:
                        description_parts.append(item['name'])
                
                if summary == '':
                    continue
                
                event = {
                    'summary': summary,
                    'description': description_parts,
                    'uid': str(uuid.uuid4()),
                    'dtstamp': datetime.now(),
                    'transp': 'OPAQUE'
                }
                
                entry_date = datetime.fromisoformat(entry['day']).date()
                
                if include_time and event_time:
                    event_datetime = datetime.combine(entry_date, event_time)
                    event['dtstart'] = event_datetime
                    
                    if menu_type.lower() == "breakfast":
                        duration = timedelta(minutes=30)
                    else:
                        duration = timedelta(minutes=45)
                    
                    end_time = event_datetime + duration
                    event['dtend'] = end_time
                else:
                    event['dtstart'] = entry_date
                
                event_list.append(event)
            except KeyError:
                continue

        return event_list

    def combine_calendars(self, calendar_list: list) -> list:
        """Combine multiple calendars into a single calendar."""
        combined_events = []
        for cal in calendar_list:
            combined_events.extend(cal)
        return combined_events

    def calendar(self, cal_events: list) -> list:
        """Generate a calendar from events."""
        return cal_events

    def ical(self, events: list) -> str:
        """Get the menu as a properly formatted iCal string."""
        lines = [
            "BEGIN:VCALENDAR",
            "VERSION:2.0",
            "PRODID:-//My School Menus//EN",
            "CALSCALE:GREGORIAN",
            "METHOD:PUBLISH"
        ]
        
        for event in events:
            lines.append("BEGIN:VEVENT")
            lines.append(f"SUMMARY:{event['summary']}")
            
            if isinstance(event['dtstart'], datetime):
                dt_str = event['dtstart'].strftime("%Y%m%dT%H%M%S")
                lines.append(f"DTSTART:{dt_str}")
            else:
                dt_str = event['dtstart'].strftime("%Y%m%d")
                lines.append(f"DTSTART;VALUE=DATE:{dt_str}")
            
            if 'dtend' in event:
                dt_str = event['dtend'].strftime("%Y%m%dT%H%M%S")
                lines.append(f"DTEND:{dt_str}")
            
            dt_str = event['dtstamp'].strftime("%Y%m%dT%H%M%SZ")
            lines.append(f"DTSTAMP:{dt_str}")
            lines.append(f"UID:{event['uid']}")
            
            description = "\\n".join(event['description'])
            lines.extend(self._fold_content("DESCRIPTION", description))
            lines.append(f"TRANSP:{event['transp']}")
            lines.append("END:VEVENT")
        
        lines.append("END:VCALENDAR")
        return "\r\n".join(lines)
    
    @staticmethod
    def _fold_content(property_name, content):
        """Properly fold long content according to iCalendar spec (75 chars)."""
        first_line = f"{property_name}:{content}"
        if len(first_line) <= 75:
            return [first_line]
            
        result = []
        current_line = first_line
        
        while len(current_line) > 75:
            result.append(current_line[:75])
            current_line = " " + current_line[75:]
        
        if current_line:
            result.append(current_line)
            
        return result

    def set_breakfast_time(self, hour: int, minute: int = 0):
        """Set the default time for breakfast events."""
        self.default_breakfast_time = time(hour, minute)

    def set_lunch_time(self, hour: int, minute: int = 0):
        """Set the default time for lunch events."""
        self.default_lunch_time = time(hour, minute)

print("✅ Calendar classes loaded successfully!")

## ⚙️ Step 5: Configuration

Fill in your school's information below. Need help finding these values? See the helper section at the bottom of this notebook.

### Finding Your School's IDs:
1. Go to [myschoolmenus.com](https://myschoolmenus.com) and find your school
2. Look at the URL - it will be like: `https://www.myschoolmenus.com/organization/DISTRICT_ID/menu/MENU_ID`
3. Extract the `DISTRICT_ID` and `MENU_ID` numbers from the URL

**Example URL:** `https://www.myschoolmenus.com/organization/2230/menu/65638`
- District ID: `2230`
- Lunch Menu ID: `65638`

In [ ]:
# Create interactive configuration form
print("🎯 Configuration Form")
print("=" * 60)

# School Information
district_id_input = widgets.IntText(
    value=2230,
    description='District ID:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='400px')
)

site_id_input = widgets.IntText(
    value=14066,
    description='Site ID:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='400px')
)

# Menu IDs
lunch_menu_id_input = widgets.IntText(
    value=65638,
    description='Lunch Menu ID:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='400px')
)

breakfast_menu_id_input = widgets.IntText(
    value=90143,
    description='Breakfast Menu ID:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='400px')
)

breakfast_enabled = widgets.Checkbox(
    value=True,
    description='Enable Breakfast Menu',
    style={'description_width': '150px'}
)

# Time Settings
include_time_input = widgets.Checkbox(
    value=True,
    description='Include Time in Events',
    style={'description_width': '150px'}
)

breakfast_hour = widgets.IntSlider(
    value=8,
    min=0,
    max=23,
    step=1,
    description='Breakfast Hour:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='400px')
)

breakfast_minute = widgets.IntSlider(
    value=0,
    min=0,
    max=59,
    step=5,
    description='Breakfast Minute:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='400px')
)

lunch_hour = widgets.IntSlider(
    value=12,
    min=0,
    max=23,
    step=1,
    description='Lunch Hour:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='400px')
)

lunch_minute = widgets.IntSlider(
    value=0,
    min=0,
    max=59,
    step=5,
    description='Lunch Minute:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='400px')
)

# Output Options
create_separate_files = widgets.Checkbox(
    value=False,
    description='Create Separate Files (breakfast/lunch)',
    style={'description_width': '250px'}
)

create_combined_file = widgets.Checkbox(
    value=True,
    description='Create Combined File',
    style={'description_width': '250px'}
)

debug_mode = widgets.Checkbox(
    value=False,
    description='Debug Mode (verbose output)',
    style={'description_width': '250px'}
)

# Display the form
display(HTML("<h3>🏫 School Information</h3>"))
display(district_id_input)
display(site_id_input)

display(HTML("<h3>🍽️ Menu IDs</h3>"))
display(lunch_menu_id_input)
display(breakfast_enabled)
display(breakfast_menu_id_input)

display(HTML("<h3>⏰ Time Settings</h3>"))
display(include_time_input)
display(breakfast_hour)
display(breakfast_minute)
display(lunch_hour)
display(lunch_minute)

display(HTML("<h3>📁 Output Options</h3>"))
display(create_combined_file)
display(create_separate_files)
display(debug_mode)

print("\n✅ Configuration form ready! Scroll down and run the next cell to generate calendars.")

## 🚀 Step 6: Generate Calendar Files

Run this cell to fetch menu data and generate your calendar files. You'll see detailed progress as the process runs.

In [ ]:
# Extract configuration values
DISTRICT_ID = district_id_input.value
SITE_ID = site_id_input.value
LUNCH_MENU_ID = lunch_menu_id_input.value
BREAKFAST_MENU_ID = breakfast_menu_id_input.value if breakfast_enabled.value else None
INCLUDE_TIME = include_time_input.value
BREAKFAST_TIME = time(breakfast_hour.value, breakfast_minute.value)
LUNCH_TIME = time(lunch_hour.value, lunch_minute.value)
CREATE_SEPARATE_FILES = create_separate_files.value
CREATE_COMBINED_FILE = create_combined_file.value
DEBUG = debug_mode.value

# Storage for generated files
generated_files = {}

print("="*80)
print("🍎 MY SCHOOL MENUS - CALENDAR GENERATOR")
print("="*80)
print(f"\n📋 Configuration Summary:")
print(f"   District ID: {DISTRICT_ID}")
print(f"   Site ID: {SITE_ID}")
print(f"   Lunch Menu ID: {LUNCH_MENU_ID}")
print(f"   Breakfast Menu ID: {BREAKFAST_MENU_ID if BREAKFAST_MENU_ID else 'Not enabled'}")
print(f"   Include Times: {INCLUDE_TIME}")
if INCLUDE_TIME:
    print(f"   Breakfast Time: {BREAKFAST_TIME.strftime('%I:%M %p')}")
    print(f"   Lunch Time: {LUNCH_TIME.strftime('%I:%M %p')}")
print(f"   Create Combined File: {CREATE_COMBINED_FILE}")
print(f"   Create Separate Files: {CREATE_SEPARATE_FILES}")
print(f"\n{'='*80}\n")

try:
    # Initialize API and Calendar objects
    print("🔧 Initializing API client and calendar generator...")
    menus = Menus()
    cal = Calendar(
        default_breakfast_time=BREAKFAST_TIME,
        default_lunch_time=LUNCH_TIME
    )
    print("   ✅ Initialization complete\n")
    
    # Process lunch menu if specified
    lunch_available_dates = []
    if LUNCH_MENU_ID:
        print(f"📥 Fetching lunch menu metadata...")
        print(f"   URL: https://myschoolmenus.com/api/organizations/{DISTRICT_ID}/menus/{LUNCH_MENU_ID}")
        try:
            lunch_menu = menus.get(district_id=DISTRICT_ID, menu_id=LUNCH_MENU_ID)
            lunch_available_dates = menus.menu_months(lunch_menu)
            print(f"   ✅ Found {len(lunch_available_dates)} months with lunch menus")
            print(f"   📅 Available months: {', '.join(d.strftime('%Y-%m') for d in lunch_available_dates)}")
        except Exception as e:
            print(f"   ❌ Error getting lunch menu: {e}")
        print()
    
    # Process breakfast menu if specified
    breakfast_available_dates = []
    if BREAKFAST_MENU_ID:
        print(f"📥 Fetching breakfast menu metadata...")
        print(f"   URL: https://myschoolmenus.com/api/organizations/{DISTRICT_ID}/menus/{BREAKFAST_MENU_ID}")
        try:
            breakfast_menu = menus.get(district_id=DISTRICT_ID, menu_id=BREAKFAST_MENU_ID)
            breakfast_available_dates = menus.menu_months(breakfast_menu)
            print(f"   ✅ Found {len(breakfast_available_dates)} months with breakfast menus")
            print(f"   📅 Available months: {', '.join(d.strftime('%Y-%m') for d in breakfast_available_dates)}")
        except Exception as e:
            print(f"   ❌ Error getting breakfast menu: {e}")
        print()
    
    # Get unique dates from both menus
    all_dates = sorted(set(lunch_available_dates + breakfast_available_dates))
    print(f"📊 Processing {len(all_dates)} total unique months\n")
    print("="*80)
    
    # Process each month
    for idx, date in enumerate(all_dates, 1):
        print(f"\n📆 Processing Month {idx}/{len(all_dates)}: {date.strftime('%B %Y')} ({date.strftime('%Y-%m')})")
        print("-"*80)
        
        all_events = []  # Will hold all events for the month
        
        # Process lunch menu for this date if available
        if LUNCH_MENU_ID and date in lunch_available_dates:
            print(f"\n🍽️  Fetching LUNCH menu data...")
            try:
                lunch_calendar_menu = menus.get(
                    district_id=DISTRICT_ID, menu_id=LUNCH_MENU_ID, date=date
                )
                print(f"   ✅ Data retrieved successfully")
                
                print(f"   🔄 Converting to calendar events...")
                lunch_events = cal.events(
                    lunch_calendar_menu, 
                    menu_type="lunch", 
                    include_time=INCLUDE_TIME
                )
                print(f"   ✅ Generated {len(lunch_events)} lunch events")
                
                if DEBUG and lunch_events:
                    print(f"   🔍 Sample event: {lunch_events[0]['summary']}")
                
                all_events.extend(lunch_events)
                
                # Write separate lunch file if requested
                if CREATE_SEPARATE_FILES and lunch_events:
                    lunch_calendar = cal.calendar(lunch_events)
                    filename = f"{date.year}-{date.month:02}-lunch-menu.ics"
                    lunch_ical = cal.ical(lunch_calendar)
                    generated_files[filename] = lunch_ical
                    print(f"   💾 Created separate lunch file: {filename}")
                    
            except Exception as e:
                print(f"   ❌ Error processing lunch menu: {e}")
        
        # Process breakfast menu for this date if available
        if BREAKFAST_MENU_ID and date in breakfast_available_dates:
            print(f"\n🥐 Fetching BREAKFAST menu data...")
            try:
                breakfast_calendar_menu = menus.get(
                    district_id=DISTRICT_ID, menu_id=BREAKFAST_MENU_ID, date=date
                )
                print(f"   ✅ Data retrieved successfully")
                
                print(f"   🔄 Converting to calendar events...")
                breakfast_events = cal.events(
                    breakfast_calendar_menu, 
                    menu_type="breakfast", 
                    include_time=INCLUDE_TIME
                )
                print(f"   ✅ Generated {len(breakfast_events)} breakfast events")
                
                if DEBUG and breakfast_events:
                    print(f"   🔍 Sample event: {breakfast_events[0]['summary']}")
                
                all_events.extend(breakfast_events)
                
                # Write separate breakfast file if requested
                if CREATE_SEPARATE_FILES and breakfast_events:
                    breakfast_calendar = cal.calendar(breakfast_events)
                    filename = f"{date.year}-{date.month:02}-breakfast-menu.ics"
                    breakfast_ical = cal.ical(breakfast_calendar)
                    generated_files[filename] = breakfast_ical
                    print(f"   💾 Created separate breakfast file: {filename}")
                    
            except Exception as e:
                print(f"   ❌ Error processing breakfast menu: {e}")
        
        # Create combined calendar file for this month
        if all_events and CREATE_COMBINED_FILE:
            print(f"\n📋 Creating COMBINED calendar file...")
            combined_calendar = cal.calendar(all_events)
            filename = f"{date.year}-{date.month:02}-combined-menu.ics"
            
            print(f"   🔄 Combining {len(all_events)} total events")
            ical = cal.ical(combined_calendar)
            generated_files[filename] = ical
            print(f"   ✅ Combined calendar file created: {filename}")
        elif not all_events:
            print(f"\n⚠️  No menu data available for {date.year}-{date.month}, skipping file creation.")
        
        print("-"*80)
    
    print(f"\n\n{'='*80}")
    print(f"✅ GENERATION COMPLETE!")
    print(f"{'='*80}")
    print(f"\n📁 Generated {len(generated_files)} calendar file(s):")
    for filename in sorted(generated_files.keys()):
        file_size = len(generated_files[filename])
        print(f"   • {filename} ({file_size:,} bytes)")
    print(f"\n💡 Run the next cell to download your calendar files!\n")
    
except Exception as e:
    print(f"\n❌ FATAL ERROR: {e}")
    print(f"\nPlease check your configuration and try again.")
    import traceback
    if DEBUG:
        print("\n🔍 Debug traceback:")
        traceback.print_exc()

## 📥 Step 7: Download Calendar Files

Run this cell to download all generated calendar files. If you have multiple files, they'll be packaged in a ZIP file for convenience.

In [ ]:
if not generated_files:
    print("❌ No files to download. Please run the generation cell first.")
else:
    print(f"📦 Preparing {len(generated_files)} file(s) for download...\n")
    
    if len(generated_files) == 1:
        # Single file - download directly
        filename = list(generated_files.keys())[0]
        content = generated_files[filename]
        print(f"📄 Downloading: {filename}")
        files.download(filename)
        
        # Write the file so it can be downloaded
        with open(filename, 'w', newline='') as f:
            f.write(content)
        files.download(filename)
        
    else:
        # Multiple files - create a ZIP
        zip_filename = f"school-menus-{datetime.now().strftime('%Y%m%d')}.zip"
        print(f"📦 Creating ZIP file: {zip_filename}\n")
        
        # Create ZIP file in memory
        zip_buffer = io.BytesIO()
        with zipfile.ZipFile(zip_buffer, 'w', zipfile.ZIP_DEFLATED) as zip_file:
            for filename, content in generated_files.items():
                print(f"   Adding: {filename}")
                zip_file.writestr(filename, content)
        
        # Write ZIP to file
        with open(zip_filename, 'wb') as f:
            f.write(zip_buffer.getvalue())
        
        print(f"\n✅ ZIP file created!")
        print(f"📥 Downloading: {zip_filename}\n")
        files.download(zip_filename)
    
    print("\n" + "="*80)
    print("✅ DOWNLOAD COMPLETE!")
    print("="*80)
    print("\n📱 Next Steps:")
    print("   1. Locate the downloaded file(s) on your computer")
    print("   2. Import the .ics file(s) into your calendar app:")
    print("      • Google Calendar: Settings → Import & Export → Import")
    print("      • Apple Calendar: File → Import")
    print("      • Outlook: File → Open & Export → Import/Export")
    print("   3. The menu events will appear on the dates they're served!\n")

## 🔍 Help: Finding Your School's IDs

### Method 1: From the Website URL

1. Go to [myschoolmenus.com](https://myschoolmenus.com)
2. Search for and navigate to your school's menu
3. Look at the URL in your browser's address bar

**URL Format:**
```
https://www.myschoolmenus.com/organization/DISTRICT_ID/menu/MENU_ID
```

**Example:**
```
https://www.myschoolmenus.com/organization/2230/menu/65638
                                          ^^^^        ^^^^^  
                                     District ID   Menu ID
```

### Finding Different Menu IDs

Many schools have separate menus for breakfast and lunch:

1. **Lunch Menu**: Click on "Lunch" on the school's page
   - URL: `https://www.myschoolmenus.com/organization/2230/menu/65638`
   - Lunch Menu ID: `65638`

2. **Breakfast Menu**: Click on "Breakfast" on the school's page
   - URL: `https://www.myschoolmenus.com/organization/2230/menu/90143`
   - Breakfast Menu ID: `90143`

### Troubleshooting

**Problem:** "No menu found" error
- ✅ Double-check your District ID and Menu ID
- ✅ Make sure you're using the correct menu ID for breakfast vs. lunch
- ✅ Try visiting the URL manually to confirm it exists

**Problem:** Calendar files are empty
- ✅ Check if menus are published for the current/upcoming months
- ✅ Some schools only publish menus during the school year

**Problem:** Import fails in calendar app
- ✅ Make sure you're importing a .ics file (not the ZIP)
- ✅ Extract the ZIP file first if you downloaded multiple months
- ✅ Try importing just one month at a time

### Need More Help?

Visit the [original repository](https://github.com/andrewdefilippis/my_school_menus) for more information and support.